# Using PISM-Cloud

PISM-Cloud is a custom AWS deployment of the Alaska Satellite Facilities' serverless, batch processing system, HyP3. PISM-Cloud allows you to run PISM simulations of Alaska glaciers by just providing a RGI glacier complex ID, PISM configuration files, and a run template.

## Connect to PISM-Cloud

In order to equitably distribute PISM-Cloud resources, users must connect to the API with NASA Earthdata Login credentials and be assigned simulation credits.

[Earthdata Login](https://urs.earthdata.nasa.gov/ "https://urs.earthdata.nasa.gov/" )
(EDL) is the authentication method used across NASA's [Earth Observation System Data Information System (EOSDIS)](https://www.earthdata.nasa.gov/about/esdis/eosdis "www.earthdata.nasa.gov/about/esdis/eosdis" ). These credentials allow you to access to any of the Earth Science data products served by EOSDIS, and for our purposes, PISM-Cloud.

There is no cost to [register for EDL credentials](https://urs.earthdata.nasa.gov/users/new "https://urs.earthdata.nasa.gov/users/new" ), and the process is quick and easy. When creating your profile, make sure to select an item in the **Study Area** field, as you may encounter access errors if that field is left blank.

You can connect to PISM-Cloud using the HyP3 SDK like:

In [1]:
import hyp3_sdk as sdk

pism_cloud = sdk.HyP3('https://pism-cloud-test.asf.alaska.edu', prompt='password')

NASA Earthdata Login username:  hyp3_pism_cloud
NASA Earthdata Login password:  ········


and check the number of credits you have with:

In [2]:
pism_cloud.check_credits()

1000

## Selecting Glacier Complexes

In the PISM-Cloud data bucket, we provide a GeoPackage of all the RGI glacier complexes, which you can use to select the region(s) you'd like to simulate.

In [3]:
import fsspec
import geopandas as gpd

rgi_file = 'rgi_c.gpkg'

fs = fsspec.filesystem('s3', anon=True)
fs.get('s3://pism-cloud-data/glacier/rgi/rgi_c.gpkg', rgi_file)

gdf = gpd.read_file(rgi_file)

ERROR 1: PROJ: proj_create_from_database: Open of /home/jhkennedy/miniforge3/envs/pism-terra/share/proj failed


We can, for example, get the IDs for all glacier complexes in Alaska larger than 500 km^2.

In [4]:
gdf[(gdf["o1region"] == "01") & (gdf["area_km2"] >= 500)].rgi_id

173     RGI2000-v7.0-C-01-01407
476     RGI2000-v7.0-C-01-03102
520     RGI2000-v7.0-C-01-03383
584     RGI2000-v7.0-C-01-03718
641     RGI2000-v7.0-C-01-04024
695     RGI2000-v7.0-C-01-04374
908     RGI2000-v7.0-C-01-06260
1153    RGI2000-v7.0-C-01-08012
1167    RGI2000-v7.0-C-01-08153
1207    RGI2000-v7.0-C-01-08332
1364    RGI2000-v7.0-C-01-09429
1700    RGI2000-v7.0-C-01-11818
1833    RGI2000-v7.0-C-01-12784
2215    RGI2000-v7.0-C-01-14612
2288    RGI2000-v7.0-C-01-14907
2303    RGI2000-v7.0-C-01-14978
2503    RGI2000-v7.0-C-01-16008
2530    RGI2000-v7.0-C-01-16106
Name: rgi_id, dtype: object

However, for this tutorial, we'll just run a small ensemble simulation of the Wrangell glacier complex.

In [5]:
rgi_ids = [
    'RGI2000-v7.0-C-01-04374',
]

## Preparing ensemble simulations

Simulation "jobs" are requested in two steps. First, we need to prepare all the necessary input data for the glacier complexes we want to simulate by submitting a PISM_TERRA_PREP_ENSEMBLE job using the template below. These jobs will create the simulation grids; project the input DEM, surface elevations, velocities, etc., onto the grids; and generate executable scripts that will run the simulation.

In [6]:
# Commented out lines will be set when we prepare the Pism-Cloud AWS job
STAGE_TEMPLATE =     {
    # "name": "RGI2000-v7.0-C-01-04374_era5_agu_1year",
    "job_type": "PISM_TERRA_PREP_ENSEMBLE",
    "job_parameters": {
        # "rgi_id": "RGI2000-v7.0-C-01-04374",
        "pism_config": "s3://pism-cloud-data/glacier/config/era5_ec2_1year.toml",
        "uq_config": "s3://pism-cloud-data/glacier/uq/era5_pdd.toml",
        "ntasks": 32,
    }
}

The `pism_config` and `uq_config` files can be either HTTPS URLs or S3 URIs. We've provided some examples in the public `s3://pism-cloud-data` bucket.

Then we'll loop through all the RGIs we want to simulate, set the `rgi_id` job parameter, and give the job an appropriate name so we can easily find it later.

In [7]:
from copy import deepcopy
from pathlib import Path


prepared_jobs = []
for rgi_id in rgi_ids:
    job_dict = deepcopy(STAGE_TEMPLATE)
    job_dict['job_parameters']['rgi_id'] = rgi_id
    job_dict['name'] = f'{rgi_id}_{Path(job_dict["job_parameters"]["pism_config"]).stem}'
    prepared_jobs.append(job_dict)

In [8]:
jobs = pism_cloud.submit_prepared_jobs(prepared_jobs)
print(jobs)

job_names = {job.name for job in jobs}
print(job_names)

1 HyP3 Jobs: 0 succeeded, 0 failed, 0 running, 1 pending.
{'RGI2000-v7.0-C-01-04374_era5_ec2_1year'}


We can use the `watch` method to periodically check on our jobs and tell us when they have finished.

In [11]:
jobs = pism_cloud.watch(jobs)

  0%|          | 0/1 [timeout in 10800 s]

## Execute the PISM-Cloud simulation

The prepare jobs will have created a set of simulations run scripts, which we need to find and then tell PISM-Cloud to execute. All job outputs are uploaded to a "content" bucket under a `job_id/rgi_id` prefix (folder). This function will look in the prefix for all run scripts and return a list of their paths.

In [12]:
import s3fs

PISM_CLOUD_BUCKET = 'hyp3-pism-cloud-test-contentbucket-zs9dctrqrlvx'

def get_run_scripts(job: sdk.Job) ->  list[str]:
    fs = s3fs.S3FileSystem(anon=True)
    files = fs.ls(f'{PISM_CLOUD_BUCKET}/{job.job_id}/{job.job_parameters["rgi_id"]}/run_scripts')
    return [str(Path(file).relative_to(f'{PISM_CLOUD_BUCKET}/{job.job_id}/')) for file in files]


And now, using the execute template, we can run each simulation in the emsemble.

In [13]:
EXECUTE_TEMPLATE = {
    # "name": "RGI2000-v7.0-C-01-09429_era5_agu_1year",
    "job_type": "PISM_TERRA_EXECUTE",
    "job_parameters": {
        # "ensemble_job_id": "042ffcdc-2134-4b18-b1af-b22fdf7cbb52",
        # "run_script": "RGI2000-v7.0-C-01-09429/run_scripts/submit_g400m_RGI2000-v7.0-C-01-09429_id_0_1978-01-01_1979-01-01.sh",
    }
}

prepared_jobs = []
for job in jobs:
    run_scripts = get_run_scripts(job)
    for script in run_scripts:
        job_dict = deepcopy(EXECUTE_TEMPLATE)
        job_dict['name'] = job.name
        job_dict['job_parameters']['ensemble_job_id'] = job.job_id
        job_dict['job_parameters']['run_script'] = script
        prepared_jobs.append(job_dict)


In [14]:
jobs = pism_cloud.submit_prepared_jobs(prepared_jobs)
print(jobs)

job_names = {job.name for job in jobs}
print(job_names)

9 HyP3 Jobs: 0 succeeded, 0 failed, 0 running, 9 pending.
{'RGI2000-v7.0-C-01-04374_era5_ec2_1year'}


In [15]:
jobs = pism_cloud.watch(jobs)

  0%|          | 0/9 [timeout in 10800 s]

In [16]:
print(jobs)

9 HyP3 Jobs: 9 succeeded, 0 failed, 0 running, 0 pending.
